# 01 · Load, clean and validate the KTAS triage data

Vitals model, step 1. The data is the Kaggle *Emergency Service – Triage Application* dataset
(1,267 adult visits to two Korean emergency departments, Oct 2016 – Sep 2017).
The target is `KTAS_expert`, the urgency level (1 = most urgent … 5 = least) agreed by three triage experts.

Source paper (cite this in the report): Moon S-H, Shim JL, Park K-S, Park C-S. *Triage accuracy and causes of
mistriage using the Korean Triage and Acuity Scale.* PLoS One 2019;14(9):<NUS-ID>.

All cleaning logic lives in `../src/ktas_data.py`, so the app and later notebooks reuse exactly the same steps.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
import ktas_data as kd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 110)

## 1. Load the raw file as text

Everything is read as text first, so nothing is silently converted. The file is semicolon-separated,
Turkish Windows encoding (`cp1254`), and uses decimal commas.

In [ ]:
raw = kd.load_raw()
print(raw.shape)
raw.head()

### What is wrong with the raw file

Problems found by auditing every column:

| Problem | Where | Fix |
|---|---|---|
| Missing vitals written as `??` | SBP, DBP, HR, RR, BT, SpO2, chief complaint | set to missing |
| Missing pain scores written as `#BOŞ!` (Turkish Excel's "#EMPTY!") | NRS_pain | set to missing, then 0 if the patient reported no pain |
| Trailing dot (`36.`) and trailing space (`0 `) | DBP, BT, Error_group, mistriage | stripped before converting to numbers |
| Decimal comma (`5,00`) | KTAS duration | converted to a decimal point |
| 5 visits recorded twice, identical except for a second diagnosis | 10 rows | merged into one row; diagnoses joined with ` \| ` |
| SpO2 = 20% in an alert patient triaged KTAS 3 | 1 row | outside plausible limits, set to missing |

In [ ]:
placeholder_counts = pd.DataFrame({p: (raw == p).sum() for p in kd.PLACEHOLDERS})
placeholder_counts[placeholder_counts.sum(axis=1) > 0]

## 2. Clean

`clean()` returns the cleaned table and a log of every change, so each fix can be reported.

In [ ]:
df, log = kd.clean(raw)
print(f"{len(raw)} raw rows -> {len(df)} visits, {df.shape[1]} columns")
log

## 3. Validate

Every check should pass. A check marked `warning` is a known gap that is kept on purpose.

In [ ]:
report = kd.validate(df)
errors = report[(report.severity == "error") & ~report.passed]
assert errors.empty, errors
report

## 4. Missing data is not random

About half of KTAS 1 patients have no vitals recorded, compared with under 1% for KTAS 3–4.
SpO2 is missing for over half of all patients.

In [ ]:
kd.missingness(df)

Most KTAS 1 patients with no vitals were in cardiac arrest, unresponsive or post-CPR, too unwell to measure.

**Consequence for the vitals model:** NEWS2 normally scores a missing parameter as 0, which would make a cardiac-arrest
patient look *low risk*. The vitals model must report missing values explicitly, and treat
"unresponsive with vitals missing" as danger, never as normal.

In [ ]:
cols = ["visit_id", "chief_complaint", "mental_label", "sbp", "hr", "rr", "spo2",
        "n_vitals_missing", "diagnosis", "disposition_label"]
df.loc[(df.ktas_expert == 1) & (df.n_vitals_missing >= 4), cols]

## 5. Summary of the cleaned data

In [ ]:
df[kd.VITALS + ["nrs_pain", "age"]].describe().T.round(1)

In [ ]:
counts = df.ktas_expert.value_counts().sort_index()
pd.DataFrame({"visits": counts, "%": (counts / len(df) * 100).round(1)}).rename_axis("KTAS_expert")

KTAS 1 has only 25 visits, so any train/test split must be **stratified** by `ktas_expert`.

Nurse vs expert triage (rows = nurse, columns = expert). Above the diagonal = nurse under-triaged (rated less urgent than the experts).

In [ ]:
pd.crosstab(df.ktas_nurse, df.ktas_expert, margins=True)

## 6. Save

In [ ]:
out = kd.save(df, log)
print("Saved to", out.resolve())
print(sorted(p.name for p in out.iterdir()))

## Which columns may be used

- **Model inputs** (known at the triage desk): `kd.TRIAGE_FEATURES`
- **Target:** `ktas_expert`
- **Never use as inputs** (known only after triage, or another version of the answer): `kd.LEAKAGE_COLS`.
  `ktas_nurse` is kept only as the human benchmark.

**Next notebook:** stratified train/validation/test split, then the NEWS2 calculator, evaluated against `ktas_expert`.

In [ ]:
print("Inputs :", kd.TRIAGE_FEATURES)
print("Target :", kd.TARGET)
print("Leakage:", kd.LEAKAGE_COLS)